In [1]:
# mike babb
# created: 2026 08 23
# updated: 2026 09 23
# find five groups of five letters

In [2]:
# standard
import math
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [3]:
import pandas as pd
import numpy as np

In [4]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [5]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [6]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


## DEMONSTRATE BITWISE OPERATIONS

In [17]:
vibex = byte_encode_words('vibex')
glyph = byte_encode_words('glyph')
muntz = byte_encode_words('muntz')
dwarf = byte_encode_words('dwarf')
jocks = byte_encode_words('jocks')
cramp = byte_encode_words('cramp')

In [18]:
# this is not equal to zero because letters are reused
((vibex | glyph) | muntz | dwarf) & cramp

167937

In [19]:
# this is equal to zero
(vibex | glyph | muntz | dwarf) & jocks

0

In [20]:
# order of operations
(vibex | glyph) &  (muntz | dwarf) 

0

In [22]:
vibex | glyph | muntz | dwarf | jocks

67043327

In [23]:
byte_encode_words('vibexglyphmuntzdwarfjocks')

67043327

# BUILD LEVEL 2 BY COMBINING TWO WORDS

In [24]:
l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)
row_index = 0
found_values = set()
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common, compute the bitwise or to add the words together
        l2 = w1_be | w2_be                          
        
        l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
        found_values.add(l2)
        row_index += 1

# trim the data frame
l2_list = l2_list[:row_index, :]
print(l2_list.shape)
l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])

(3213696, 3)


# BUILD LEVELS 4 AND 5 BY COMBINING TWO ITEMS FROM THE L2 LIST AND THEN
# COMPARING THAT WITH THE WORD BYTE ARRAY ONE MORE TIME

In [27]:
l2_df_test = l2_df.drop_duplicates(subset = 'l2').reset_index(drop = True)

In [28]:
l2_all = l2_df_test['l2'].to_numpy(dtype = np.int32)

In [32]:
# assign words
l2_df_test['w1'] = l2_df_test['w1b'].map(word_byte_to_word_dict)
l2_df_test['w2'] = l2_df_test['w2b'].map(word_byte_to_word_dict)

In [33]:
# let's just use the word jocks
w_l2_df_test = l2_df_test.loc[(l2_df_test['w1'] == 'jocks') |
                              (l2_df_test['w2'] == 'jocks'), ['w1b', 'w2b', 'l2']].reset_index(drop = True)

In [34]:
w_l2_df_test.shape

(928, 3)

In [35]:
w_l2_df_test['l2'].unique().shape

(928,)

In [38]:
# so, now, let's try computing all possible pairs
start_pos = 0
total_output = np.zeros(shape = (100000000, 5), dtype = np.int32)
for i_row, row in w_l2_df_test.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # compare the current l2 to all l2 - this will find all instances
    # a value of zero indicates that there are no letters in common
    # indexer for l2, l3, and l4
    positional_idx_l2l3l4 = (l2_all & l2) == 0
    if positional_idx_l2l3l4.any():

        # l2 words and words with different letters
        output_array_w3bw4b = l2_all[positional_idx_l2l3l4]    
        
        # l2, l3, l4 accumulated letters
        output_array_l2l3l4 = output_array_w3bw4b | l2

        # create the temp output
        n_rows_l2l3l4 = output_array_l2l3l4.shape[0]    

        # check against the word_byte_array for w5b
        
        for w3bw4b, l2l3l4 in zip(output_array_w3bw4b, output_array_l2l3l4):

            positional_idx_l2l3l4l5 = (l2l3l4 & word_byte_array) == 0
            if positional_idx_l2l3l4l5.any(): 
                output_array_l2l3l4l5 = word_byte_array[positional_idx_l2l3l4l5]
                
                n_rows_l2l3l4l5 = output_array_l2l3l4l5.shape[0]
            
                temp_output = np.zeros(shape = (n_rows_l2l3l4l5, 5), dtype = np.int32)
                temp_output[:, 0] = w1b
                temp_output[:, 1] = w2b

                temp_output[:, 2] = l2

                # calculate w3b and w4b
                # this is the 'other' w1b and w2b values                
                temp_output[:, 3] = w3bw4b
                #w5b
                temp_output[:, 4] = output_array_l2l3l4l5
        
                total_output[start_pos:start_pos + n_rows_l2l3l4l5, :] = temp_output
        
                start_pos += n_rows_l2l3l4l5        
    
    if i_row % 1000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row)   

0


# CREATE AND SHAPE THE OUTPUT

In [57]:
output = total_output[:start_pos]

In [58]:
output.shape

(67, 5)

In [59]:
# turn it into a dataframe
output_df = pd.DataFrame(data = output, columns = ['w1b', 'w2b', 'l2',  'l3l4', 'w5b'])

In [60]:
output_df.head()

,w1b,w2b,l2,l3l4,w5b
0,33562898,280068,33842966,27920489,5279872
1,33562898,280068,33842966,14194856,19005505
2,33562898,280068,33842966,24285377,8914984
3,33685523,280068,33965591,27797864,5279872
4,33685523,280068,33965591,14194856,18882880


In [61]:
output_df.head()

,w1b,w2b,l2,l3l4,w5b
0,33562898,280068,33842966,27920489,5279872
1,33562898,280068,33842966,14194856,19005505
2,33562898,280068,33842966,24285377,8914984
3,33685523,280068,33965591,27797864,5279872
4,33685523,280068,33965591,14194856,18882880


# JOIN TO GET THE W3B AND THE W4B

In [62]:
l3l4_df = l2_df[['w1b', 'w2b', 'l2']].copy()
l3l4_df.columns = ['w3b', 'w4b', 'l3l4',]

In [63]:
l3l4_df.shape

(3213696, 3)

In [64]:
output_df = pd.merge(left = output_df, right = l3l4_df)

In [65]:
output_df.shape

(81, 7)

In [66]:
output_df.head()

,w1b,w2b,l2,l3l4,w5b,w3b,w4b
0,33562898,280068,33842966,27920489,5279872,8914984,19005505
1,33562898,280068,33842966,14194856,19005505,8914984,5279872
2,33562898,280068,33842966,24285377,8914984,19005505,5279872
3,33685523,280068,33965591,27797864,5279872,8914984,18882880
4,33685523,280068,33965591,14194856,18882880,8914984,5279872


In [67]:
# reorder...
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b']
output_df = output_df[col_names].copy()

In [68]:
# get words!
for ii in range(1, 6):
    bcn = f"w{ii}b"
    cn = f"w{ii}"
    output_df[cn] = output_df[bcn].map(word_byte_to_word_dict)

In [77]:
# count the remainder letter
lc_set = set(ascii_lowercase)
def get_remainder_letter(row):
    my_set = set()
    for cn in ['w1', 'w2', 'w3', 'w4', 'w5']:
        my_set.update(row[cn])

    return ''.join(lc_set.difference(my_set))

output_df['remaining_letter'] = output_df.apply(get_remainder_letter, axis = 1)



In [78]:
# count unique words - JUST TO VERIFY
col_names = ['w1', 'w2', 'w3', 'w4', 'w5']
output_df['n_unique_words'] = output_df[col_names].apply(lambda x: len(set(x)), axis = 1)

In [79]:
# add the words - ALSO TO VERIFY
output_df['bitwise_or'] = 0
output_df['bitwise_and'] = 0
for cn_idx in range(1, 6):
    b_cn = f"w{cn_idx}b"
    w_cn = f"w{cn_idx}"
    output_df[w_cn] = output_df[b_cn].map(word_byte_to_word_dict)
    output_df['bitwise_and'] = output_df['bitwise_and'] & output_df[b_cn]
    output_df['bitwise_or'] = output_df['bitwise_or'] | output_df[b_cn]


In [80]:
output_df.head()

,w1b,w2b,w3b,w4b,w5b,w1,w2,w3,w4,w5,n_unique_words,bitwise_or,bitwise_and,remaining_letter
0,33562898,280068,8914984,19005505,5279872,bizen,jocks,fldxt,gravy,whump,5,67043327,0,q
1,33562898,280068,8914984,5279872,19005505,bizen,jocks,fldxt,whump,gravy,5,67043327,0,q
2,33562898,280068,19005505,5279872,8914984,bizen,jocks,gravy,whump,fldxt,5,67043327,0,q
3,33685523,280068,8914984,18882880,5279872,braze,jocks,fldxt,vying,whump,5,67043327,0,q
4,33685523,280068,8914984,5279872,18882880,braze,jocks,fldxt,whump,vying,5,67043327,0,q


In [82]:
output_df.to_excel(excel_writer='test.xlsx', index = False)

In [ ]:
o_word_df.head()

In [ ]:
# so, now, let's try computing all possible pairs
total_output = np.full(shape = (1000000, 9), fill_value= -1, dtype = np.int32)
row_index = 0
for i_row, row in l2_df.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # indexer for l3
    positional_idx_l3 = (word_byte_array & l2) == 0

    # l3 words with different letters
    output_array_w3b = word_byte_array[positional_idx_l3]    

    # l3 accumulated letters
    output_array_l3 = output_array_w3b | l2

    ## enumerate level 3
    for w3b, l3 in zip(output_array_w3b, output_array_l3):

        # build level 4

        # l4 idx
        positional_idx_l4 = (word_byte_array & l3) == 0

        # words with different letters
        output_array_w4b = word_byte_array[positional_idx_l4]    
        
        # accumulated letters
        output_array_l4 = output_array_w4b | l3

        ## enumerate level 5
        for w4b, l4 in zip(output_array_w4b, output_array_l4):

            # build level 5

            # l5 idx
            positional_idx_l5 = (word_byte_array & l4) == 0
            
            # words with different letters
            output_array_w5b = word_byte_array[positional_idx_l5]    

            if output_array_w5b.size > 0:
                    
                # accumulated letters
                output_array_l5 = output_array_w5b | l4

                ## gather and combine the output
                for w5b, l5 in zip(output_array_w5b, output_array_l5):

                    temp_list = np.array([w1b, w2b, w3b, w4b, w5b, l2, l3, l4, l5], dtype = np.int32)
                    total_output[row_index, :] = temp_list                   
                    row_index += 1


    if i_row % 10000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row, row_index)
    


# CREATE AND SAVE OUTPUT

In [ ]:
total_output = total_output[:row_index, :]
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b', 'l2', 'l3', 'l4', 'l5']
l5_df = pd.DataFrame(data = total_output, columns = col_names)


In [ ]:
l5_df.shape

In [ ]:
l5_df.head()

In [ ]:
l5_df.tail()

In [ ]:
l5_df.to_csv(path_or_buf='l5.txt', sep = '\t', index = False)
